# Lab Assignment 13

**Prerequisite:** the class demo notebook *ICL_and_Instruction_Finetuning_Demo.ipynb*  
**Runtime:** Google Colab with a T4 GPU (about 10–15 minutes of GPU time per task)

Each task is a small, real NLP problem: you design the solution, run the experiments and explain what you found. Only data loading and evaluation functions are given, so that everyone's numbers are comparable.

## Rules
- Use only the models named in each task.
- You may reuse any code from the class demo. Cite any other source.
- Fix random seeds and report the settings you used.
- The two tasks are independent, so you can restart the runtime between them.

## Submission
Submit this notebook as `.ipynb` **with all outputs visible**.

## 0. Setup

In [ ]:
!pip install -q transformers datasets accelerate sentence-transformers scikit-learn

In [ ]:
import re, time, random, gc, sqlite3, warnings
from collections import Counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq,
                          Trainer, TrainingArguments)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 300)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()


def load_causal_lm(name):
    """Load a decoder-only model in fp32 (numerically safe on a T4) and its tokenizer."""
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
    return tok, model


@torch.no_grad()
def generate_batch(model, tok, prompts, max_new_tokens=64, batch_size=8, stop_strings=None, desc="generating"):
    """Greedy, batched generation. Returns only the new text for each prompt, cut at the first stop string."""
    kwargs = dict(stop_strings=stop_strings, tokenizer=tok) if stop_strings else {}
    old_side, tok.padding_side = tok.padding_side, "left"
    outputs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc=desc, leave=False):
        enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, **kwargs)
        for row in gen[:, enc["input_ids"].shape[1]:]:
            text = tok.decode(row, skip_special_tokens=True)
            for s in (stop_strings or []):
                text = text.split(s)[0]
            outputs.append(text.strip())
    tok.padding_side = old_side
    return outputs

---
# Section A: In-Context Learning

## Task 1. A news-topic classifier with zero training (50 marks)

**Scenario.** A news website wants to tag articles as **World**, **Sports**, **Business** or **Sci/Tech**. It has only 200 labelled articles and no budget to train a model. Build the best classifier you can using **only in-context learning**, then decide whether it is actually a better choice than a simple classical model.

**Constraints**
- Model: `Qwen/Qwen2.5-0.5B` (base). No weight updates.
- Demonstrations may only come from `ag_pool`, with **at most 8 per prompt**.
- Every prediction must be one of the four classes. Report final numbers on `ag_test` using `report_classification`.

**What you must do**
1. **Baselines.** Evaluate zero-shot and 8 randomly chosen demonstrations.
2. **Retrieval-based demonstrations.** For each test article, choose the 8 most similar articles from `ag_pool` as demonstrations, using sentence embeddings from `sentence-transformers/all-MiniLM-L6-v2`. Compare with random selection.
3. **Classical baseline.** Train TF-IDF + logistic regression on the same 200 articles.
4. **Error analysis.** Show the confusion matrix of your best system and explain 3 of its mistakes.

**Analysis questions** (about 250 words in total)
- (a) Did retrieval-based demonstrations beat random ones? Why do you think similar examples help?
- (b) Which classes get confused most often, and why might that be?
- (c) Compare your best ICL system with logistic regression on accuracy **and** speed (seconds per article). Which would you deploy for this website, and why?



In [ ]:
ag = load_dataset("fancyzhx/ag_news")
AG_LABELS = ["World", "Sports", "Business", "Sci/Tech"]

ag_test = ag["test"].shuffle(seed=0).select(range(200))
_train = ag["train"].shuffle(seed=1)
ag_pool = _train.select(range(200))          # the ONLY labelled data you may use
ag_dev = _train.select(range(200, 300))      # use for design decisions (prompt format, label words)

for name, ds in [("pool", ag_pool), ("dev", ag_dev), ("test", ag_test)]:
    counts = Counter(ds["label"])
    print(f"{name:5s}", {AG_LABELS[k]: counts[k] for k in range(4)})
print("\nExample:", ag_pool[0])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

RESULTS_T1 = []   # append the dict returned by report_classification for every system


def report_classification(y_true, y_pred, method, sec_per_example=None, show_cm=False):
    """Fixed evaluation for Task 1. y_true / y_pred are lists of label ids (0-3)."""
    res = {"method": method,
           "accuracy": accuracy_score(y_true, y_pred),
           "macro_f1": f1_score(y_true, y_pred, average="macro"),
           "sec_per_example": sec_per_example}
    if show_cm:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=list(range(4)), display_labels=AG_LABELS,
                                                cmap="Blues", xticks_rotation=45)
        plt.title(method); plt.show()
    return res

### Task 1: Your solution
Add as many code and text cells as you need.

#### Approach

All design decisions (prompt template, label words, how demonstrations are chosen) are made on `ag_dev`, so `ag_test` is only used for the final numbers.

| Setting | Value |
|---|---|
| Random seed | `SEED = 0` for sampling; extra seeds 1 and 2 for random demonstrations |
| ICL model | `Qwen/Qwen2.5-0.5B` (base, no weight updates; weights in the dtype printed below, label scores computed in fp32) |
| Demonstrations | at most `K = 8` per prompt, only from `ag_pool` |
| Retrieval embeddings | `sentence-transformers/all-MiniLM-L6-v2`, cosine similarity |
| Batch size for inference | 8 |

**Guaranteeing one of the four classes.** Rather than generating free text and parsing it, I read the model's next-token distribution at the end of the prompt and compare only the first token of each of the four label words. The prediction is the highest of those four, so every output is a valid class. This is the same idea as `word_logprob` in the class demo, done in batches. The code asserts that the four first tokens are different.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SEED, K, ICL_BATCH = 0, 8, 8
set_seed(SEED)

tok_q, model_q = load_causal_lm("Qwen/Qwen2.5-0.5B")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

pool_texts, pool_labels = ag_pool["text"], ag_pool["label"]
dev_texts, dev_labels = ag_dev["text"], ag_dev["label"]
test_texts, test_labels = ag_test["text"], ag_test["label"]


def embed(texts):
    return embedder.encode(texts, batch_size=64, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)


pool_emb = embed(pool_texts)
print("Pool embeddings:", pool_emb.shape)
print("Qwen weights dtype:", model_q.dtype, "| device:", device)

#### Prompt building and label scoring
`TEMPLATES` and `LABEL_WORDS` are the design options tried on `ag_dev`. A demonstration is the article followed by its label, and the prompt stops just before the label of the query article.

In [ ]:
TEMPLATES = {
    "plain":       ("", "Article: {text}\nTopic:"),
    "instruction": ("Classify each news article into one of these topics: {labels}.\n\n", "Article: {text}\nTopic:"),
    "news":        ("", "News: {text}\nCategory:"),
}
LABEL_WORDS = {
    "as_given":   ["World", "Sports", "Business", "Sci/Tech"],
    "technology": ["World", "Sports", "Business", "Technology"],
    "science":    ["International", "Sports", "Business", "Science"],
}


def build_prompt(text, demo_ids, template, words):
    header, body = TEMPLATES[template]
    head = header.format(labels=", ".join(words))
    shots = "".join(body.format(text=pool_texts[i]) + " " + words[pool_labels[i]] + "\n\n" for i in demo_ids)
    return head + shots + body.format(text=text)


def label_token_ids(words):
    ids = [tok_q.encode(" " + w, add_special_tokens=False)[0] for w in words]
    assert len(set(ids)) == len(words), f"label words share a first token: {words}"
    return ids


@torch.no_grad()
def score_labels(prompts, words, batch_size=ICL_BATCH, desc="scoring"):
    """Log-probabilities over the four classes, read from the last position of each prompt."""
    ids = label_token_ids(words)
    old_side, tok_q.padding_side = tok_q.padding_side, "left"
    out = []
    for i in tqdm(range(0, len(prompts), batch_size), desc=desc, leave=False):
        enc = tok_q(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        logits = model_q(**enc, logits_to_keep=1).logits[:, -1, ids].float()  # only the last position is needed
        out.append(torch.log_softmax(logits, dim=-1).cpu())
    tok_q.padding_side = old_side
    return torch.cat(out).numpy()


def run_icl(texts, demo_lists, template, words, desc="scoring"):
    """Returns (predictions, seconds per article, class log-probabilities)."""
    prompts = [build_prompt(t, d, template, words) for t, d in zip(texts, demo_lists)]
    t0 = time.perf_counter()
    scores = score_labels(prompts, words, desc=desc)
    return scores.argmax(1).tolist(), (time.perf_counter() - t0) / len(texts), scores


def random_demos(n, k, seed):
    rng = np.random.default_rng(seed)
    return [rng.choice(len(pool_texts), k, replace=False).tolist() for _ in range(n)]


def retrieve(query_emb, k=K, mode="top", nearest_last=True):
    """Pool indices of the demonstrations for each query.
    mode='top': the k most similar. mode='balanced': the k//4 most similar of every class."""
    sims = query_emb @ pool_emb.T
    labels = np.array(pool_labels)
    out = []
    for row in sims:
        if mode == "top":
            idx = np.argsort(-row)[:k]
        else:
            idx = np.concatenate([np.where(labels == c)[0][np.argsort(-row[labels == c])[:k // 4]] for c in range(4)])
            idx = idx[np.argsort(-row[idx])]
        out.append((idx[::-1] if nearest_last else idx).tolist())
    return out


print(build_prompt(dev_texts[0], random_demos(1, 2, SEED)[0], "instruction", LABEL_WORDS["as_given"]))

#### Design decisions on `ag_dev`
First I search over prompt template × label words, using 8 random demonstrations (and separately for zero-shot, since the best zero-shot prompt may differ). With only 100 dev articles, a gap of 1–2 points is within noise, so these choices are reasonable rather than provably optimal.

In [ ]:
dev_random = random_demos(len(dev_texts), K, SEED)
dev_zero = [[] for _ in dev_texts]


def grid_search(demo_lists, title):
    rows = []
    for t in TEMPLATES:
        for w_name, words in LABEL_WORDS.items():
            pred, _, _ = run_icl(dev_texts, demo_lists, t, words, desc=f"{title} {t}/{w_name}")
            rows.append({"template": t, "labels": w_name,
                         "dev_accuracy": accuracy_score(dev_labels, pred),
                         "dev_macro_f1": f1_score(dev_labels, pred, average="macro")})
    return pd.DataFrame(rows).sort_values(["dev_accuracy", "dev_macro_f1"], ascending=False).reset_index(drop=True)


grid_few = grid_search(dev_random, "8-shot")
grid_zero = grid_search(dev_zero, "0-shot")
print("8 random demonstrations:"); display(grid_few)
print("Zero-shot:"); display(grid_zero)

TEMPLATE, WORDS_NAME = grid_few.loc[0, "template"], grid_few.loc[0, "labels"]
WORDS = LABEL_WORDS[WORDS_NAME]
ZS_TEMPLATE, ZS_WORDS_NAME = grid_zero.loc[0, "template"], grid_zero.loc[0, "labels"]
ZS_WORDS = LABEL_WORDS[ZS_WORDS_NAME]
print(f"Chosen for few-shot: template={TEMPLATE}, labels={WORDS_NAME}")
print(f"Chosen for zero-shot: template={ZS_TEMPLATE}, labels={ZS_WORDS_NAME}")

Next, with that prompt fixed, I compare ways of choosing the demonstrations on `ag_dev`: random, the 8 nearest articles (in two orders), and a class-balanced version (the 2 nearest of each class). The balanced version guards against the case where all 8 neighbours share a label and the model simply copies it. The winner on dev becomes my "best ICL system" for the error analysis.

In [ ]:
dev_emb = embed(dev_texts)
DEV_VARIANTS = {
    "Random-8": dev_random,
    "Retrieval top-8 (nearest last)": retrieve(dev_emb, K, "top", True),
    "Retrieval top-8 (nearest first)": retrieve(dev_emb, K, "top", False),
    "Retrieval balanced 2-per-class (nearest last)": retrieve(dev_emb, K, "balanced", True),
}
rows = []
for name, demos in DEV_VARIANTS.items():
    pred, _, _ = run_icl(dev_texts, demos, TEMPLATE, WORDS, desc=name)
    rows.append({"variant": name, "dev_accuracy": accuracy_score(dev_labels, pred),
                 "dev_macro_f1": f1_score(dev_labels, pred, average="macro")})
dev_variants = pd.DataFrame(rows).sort_values(["dev_accuracy", "dev_macro_f1"], ascending=False).reset_index(drop=True)
display(dev_variants)
BEST_DEV_VARIANT = dev_variants.loc[0, "variant"]
BEST_RETRIEVAL_VARIANT = dev_variants[dev_variants["variant"].str.startswith("Retrieval")].iloc[0]["variant"]
print("Best on dev:", BEST_DEV_VARIANT, "| best retrieval variant:", BEST_RETRIEVAL_VARIANT)

#### Final runs on `ag_test`
Each system is timed from the start of its per-article work, so retrieval systems include embedding the query and searching the pool. Random demonstrations are repeated with 3 seeds to show how much the sample matters.

The last ICL row, **8-NN label vote**, is a diagnostic rather than a candidate: it skips the language model and takes a similarity-weighted vote over the labels of the same 8 retrieved articles. It shows how much of retrieval's accuracy comes from the neighbours' labels alone.

In [ ]:
TEST_PREDS, TEST_SCORES, TEST_DEMOS = {}, {}, {}


def record(name, pred, sec, scores=None, demos=None):
    TEST_PREDS[name], TEST_SCORES[name], TEST_DEMOS[name] = pred, scores, demos
    RESULTS_T1.append(report_classification(test_labels, pred, name, sec))


# Zero-shot
zero = [[] for _ in test_texts]
pred, sec, s = run_icl(test_texts, zero, ZS_TEMPLATE, ZS_WORDS, desc="zero-shot")
record("Zero-shot", pred, sec, s, zero)

# Random demonstrations, 3 seeds
for seed in [0, 1, 2]:
    demos = random_demos(len(test_texts), K, seed)
    pred, sec, s = run_icl(test_texts, demos, TEMPLATE, WORDS, desc=f"random seed {seed}")
    record(f"Random-8 (seed {seed})", pred, sec, s, demos)


def timed_retrieval(mode, nearest_last):
    t0 = time.perf_counter()
    q = embed(test_texts)
    demos = retrieve(q, K, mode, nearest_last)
    return demos, q, (time.perf_counter() - t0) / len(test_texts)


# Retrieval variants
RETRIEVAL_SPECS = {
    "Retrieval top-8 (nearest last)": ("top", True),
    "Retrieval top-8 (nearest first)": ("top", False),
    "Retrieval balanced 2-per-class (nearest last)": ("balanced", True),
}
for name, (mode, nearest_last) in RETRIEVAL_SPECS.items():
    demos, q_emb, retr_sec = timed_retrieval(mode, nearest_last)
    pred, sec, s = run_icl(test_texts, demos, TEMPLATE, WORDS, desc=name)
    record(name, pred, sec + retr_sec, s, demos)


# Diagnostic: label vote over the same 8 neighbours, no language model
t0 = time.perf_counter()
q_emb = embed(test_texts)
top = retrieve(q_emb, K, "top", False)
sims = q_emb @ pool_emb.T
vote = []
for row, idx in zip(sims, top):
    w = np.zeros(4)
    for i in idx:
        w[pool_labels[i]] += row[i]
    vote.append(int(w.argmax()))
record("8-NN label vote (diagnostic, no LLM)", vote, (time.perf_counter() - t0) / len(test_texts))

BEST_ICL = "Random-8 (seed 0)" if BEST_DEV_VARIANT == "Random-8" else BEST_DEV_VARIANT
print("Best ICL system (selected on dev):", BEST_ICL)

#### Classical baseline: TF-IDF + logistic regression
Trained on exactly the same 200 labelled articles. The regularisation strength `C` is chosen on `ag_dev`. Prediction time is measured on the CPU, which is how a cheap deployment would run it.

In [ ]:
best_C, best_dev = None, -1
for C in [1, 3, 10, 30, 100]:
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    clf = LogisticRegression(C=C, max_iter=2000, random_state=SEED)
    clf.fit(vec.fit_transform(pool_texts), pool_labels)
    acc = accuracy_score(dev_labels, clf.predict(vec.transform(dev_texts)))
    print(f"C={C:<4d} dev accuracy={acc:.3f}")
    if acc > best_dev:
        best_C, best_dev = C, acc

t0 = time.perf_counter()
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
clf = LogisticRegression(C=best_C, max_iter=2000, random_state=SEED)
clf.fit(vec.fit_transform(pool_texts), pool_labels)
fit_sec = time.perf_counter() - t0

t0 = time.perf_counter()
pred = clf.predict(vec.transform(test_texts)).tolist()
lr_sec = (time.perf_counter() - t0) / len(test_texts)
record("TF-IDF + logistic regression", pred, lr_sec)
print(f"Chosen C={best_C}. Fit time on 200 articles: {fit_sec:.3f} s")

#### Error analysis of the best ICL system
The cell below shows the confusion matrix (via the provided evaluation function) and lists every test mistake, with the model's confidence and the labels of the demonstrations it saw.

In [ ]:
_ = report_classification(test_labels, TEST_PREDS[BEST_ICL], BEST_ICL, show_cm=True)

errors = [i for i, (g, p) in enumerate(zip(test_labels, TEST_PREDS[BEST_ICL])) if g != p]
probs = np.exp(TEST_SCORES[BEST_ICL])
print(f"{len(errors)} mistakes out of {len(test_labels)}\n")
for i in errors:
    demo_labels = Counter(AG_LABELS[pool_labels[d]] for d in TEST_DEMOS[BEST_ICL][i])
    print(f"[test #{i}] gold={AG_LABELS[test_labels[i]]} | pred={AG_LABELS[TEST_PREDS[BEST_ICL][i]]} "
          f"| P(pred)={probs[i].max():.2f} | demos={dict(demo_labels)}")
    print("   ", test_texts[i][:320])
    print()

### Task 1: Results and analysis ✍️

_Results table, plots, and answers to the analysis questions go here._

In [ ]:
def paired_bootstrap(correct_a, correct_b, n=10000, seed=SEED):
    """Accuracy difference a - b with a 95% bootstrap interval, resampling test articles."""
    a, b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), (n, len(a)))
    diffs = (a[idx] - b[idx]).mean(1)
    return (a - b).mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)


table_t1 = pd.DataFrame(RESULTS_T1).sort_values("accuracy", ascending=False).reset_index(drop=True)
display(table_t1.round({"accuracy": 3, "macro_f1": 3, "sec_per_example": 5}))

random_acc = table_t1[table_t1["method"].str.startswith("Random")]["accuracy"]
print(f"Random-8 over 3 seeds: mean accuracy {random_acc.mean():.3f}, std {random_acc.std(ddof=0):.3f}")


def correct(name):
    return [g == p for g, p in zip(test_labels, TEST_PREDS[name])]


pairs = [(BEST_RETRIEVAL_VARIANT, f"Random-8 (seed {s})") for s in [0, 1, 2]] + [(BEST_ICL, "TF-IDF + logistic regression")]
rows = []
for a, b in pairs:
    d, lo, hi = paired_bootstrap(correct(a), correct(b))
    rows.append({"A": a, "B": b, "accuracy A - B": round(d, 3), "95% CI low": round(lo, 3), "95% CI high": round(hi, 3)})
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(8, 4.2))
plot_df = table_t1.sort_values("accuracy")
bars = ax.barh(plot_df["method"], plot_df["accuracy"], color="#2a78b5", height=0.62)
ax.bar_label(bars, labels=[f"{v:.3f}" for v in plot_df["accuracy"]], padding=4, fontsize=9)
ax.set_xlim(0, 1.05); ax.set_xlabel("accuracy on ag_test (200 articles)")
ax.set_title("Task 1: test accuracy by system", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", alpha=0.25); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

del model_q, embedder; clear_gpu()

---
# Section B: Instruction Fine-Tuning

## Task 2. English → SQL: prompt a ready-made model or fine-tune your own? (50 marks)

**Scenario.** A company wants its analysts to query databases in plain English. Given a table schema (a `CREATE TABLE` statement) and a question, a model must write the SQL query. Should the company **prompt** an existing instruction-tuned model, or **fine-tune** a small model on examples? Build both and recommend one.

**Data:** each example has a `question`, a `context` (the schema) and an `answer` (the gold SQL).

**Constraints**
- Prompting system: `Qwen/Qwen2.5-0.5B-Instruct` (use its chat template).
- Fine-tuned system: `openai-community/gpt2` (124M), with the loss computed on the SQL tokens only.
- Training and demonstration data come only from `sql_train_pool`. Evaluate every system on `sql_test` with `evaluate_sql`.

**What you must do**
1. **Prompting baseline.** Prompt Qwen2.5-0.5B-Instruct with 3 examples from the pool. The model may add explanations or code fences, so extract just the SQL from its output.
2. **Fine-tuning.** Design an input format for (schema, question) → SQL and fine-tune GPT-2 twice, on **500** and on **2,000** examples, with the same hyperparameters.
3. **Error analysis.** Look at 10 mistakes of your best system and label each one: wrong column, wrong condition or aggregate, wrong value, invalid SQL, or correct but written differently.

**Analysis questions** (about 250 words in total)
- (a) How did fine-tuned GPT-2 compare with prompting a model four times its size? How much did going from 500 to 2,000 examples help?
- (b) How many of your "mistakes" were actually correct queries written differently? Why does exact match underestimate performance?
- (c) Write a short recommendation to the company: prompting or fine-tuning? Consider accuracy, the labelled data needed, cost, and what happens when new tables are added.

**Marking:** prompting baseline 12 · fine-tuning (both sizes) 18 · error analysis 8 · discussion 12

In [ ]:
sql_ds = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=0)
sql_test = sql_ds.select(range(200))
sql_train_pool = sql_ds.select(range(200, 5200))   # training, validation and demonstration data

ex = sql_test[0]
print("SCHEMA  :", ex["context"]); print("QUESTION:", ex["question"]); print("GOLD SQL:", ex["answer"])

In [ ]:
def normalize_sql(q):
    """Lenient normalisation for exact match: case, whitespace, quote style, trailing semicolon."""
    q = q.strip().rstrip(";").replace('"', "'")
    q = re.sub(r"\s+", " ", q)
    q = re.sub(r"\s*([(),=<>])\s*", r"\1", q)
    return q.lower().strip()


def executes(sql, context):
    """True if the query runs without error on an empty database built from the schema."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(context)
        con.execute(sql).fetchall()
        return True
    except Exception:
        return False
    finally:
        con.close()


def evaluate_sql(preds, dataset, system):
    """Fixed evaluation for Task 2. preds: list of SQL strings aligned with dataset."""
    golds, contexts = dataset["answer"], dataset["context"]
    return {"system": system,
            "exact_match": float(np.mean([normalize_sql(p) == normalize_sql(g) for p, g in zip(preds, golds)])),
            "executable": float(np.mean([executes(p, c) for p, c in zip(preds, contexts)]))}


RESULTS_T2 = []   # append the dict returned by evaluate_sql for every system
print("Reference (gold queries):", evaluate_sql(sql_test["answer"], sql_test, "gold"))

### Task 2: Your solution
Add as many code and text cells as you need.

#### Approach

| Setting | Value |
|---|---|
| Random seed | `SEED = 0` (data subsets, demonstrations, training) |
| Prompted model | `Qwen/Qwen2.5-0.5B-Instruct`, chat template, 3 demonstrations |
| Fine-tuned model | `openai-community/gpt2`, loss on SQL tokens only (labels of prompt tokens are `-100`) |
| Pool split | first ~2,200 for training, then the rest for demonstrations, last 200 for validation |
| Fine-tuning hyperparameters | batch 16, learning rate 1e-4, 10 epochs, cosine schedule with 10% warmup, weight decay 0.01, fp16, best epoch by validation loss |

**Why a validation split?** Choices such as prompt style and input format are made on `sql_val`, the last 200 pool examples, so `sql_test` stays untouched until the final numbers. `sql_val` is never used for training or as a demonstration.

In [ ]:
import shutil

SEED = 0
set_seed(SEED)
MAX_LEN, MAX_NEW_SQL = 256, 96

sql_val = sql_train_pool.select(range(4800, 5000))
demo_region = sql_train_pool.select(range(2200, 4800))


def extract_sql(text):
    """Pull one SQL statement out of a model's reply: code fences first, then the first SELECT."""
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", text, re.S | re.I)
    if m:
        text = m.group(1)
    elif "```" in text:
        text = re.sub(r"^(?:sql|sqlite)\b", "", text.split("```", 1)[1].strip(), flags=re.I)
    m = re.search(r"(?im)^\s*SELECT\b", text) or re.search(r"\bSELECT\b", text, re.I)
    if not m:
        return " ".join(text.split())
    text = text[m.start():].strip()

    # Stop at the first semicolon that is outside a string literal
    quote = None
    for j, ch in enumerate(text):
        if quote:
            quote = None if ch == quote else quote
        elif ch in "'\"":
            quote = ch
        elif ch == ";":
            text = text[:j]
            break

    # Stop where the SQL ends and prose begins
    keep = []
    for line in text.splitlines():
        first = line.strip().split(" ")[0].upper().rstrip(",") if line.strip() else ""
        sql_start = first in {"SELECT", "FROM", "WHERE", "GROUP", "ORDER", "HAVING", "LIMIT", "JOIN", "INNER", "LEFT",
                              "RIGHT", "FULL", "CROSS", "ON", "AND", "OR", "UNION", "EXCEPT", "INTERSECT", "AS"} \
                    or first.startswith(("(", ")"))
        if not line.strip() or not (sql_start or (keep and keep[-1].rstrip().endswith((",", "(")))):
            break
        keep.append(line)
    return " ".join(" ".join(keep).split())


# Quick check of the extractor on the kinds of reply a chat model produces
for reply in ["```sql\nSELECT a FROM t;\n```\nThis query selects a.",
              "Here is the query:\n\nSELECT COUNT(*)\nFROM t\nWHERE x = 'a;b'",
              "SELECT a FROM t\n\nExplanation: it reads column a."]:
    print(repr(extract_sql(reply)))

#### 1. Prompting baseline
Two choices are compared on `sql_val`:
- **Demonstrations:** 3 picked at random, or 3 chosen to cover different query shapes (an aggregate, a filter, a join).
- **Message layout:** demonstrations as earlier chat turns, or all inside a single user message.

The system message asks for one SQLite query and nothing else, but the model can still add fences or explanations, so every reply goes through `extract_sql`.

In [ ]:
tok_i, model_i = load_causal_lm("Qwen/Qwen2.5-0.5B-Instruct")
SYSTEM = "You translate English questions into SQLite queries. Reply with exactly one SQL query and nothing else."


def user_msg(ex):
    return f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"


def chat_multi(ex, demos):
    msgs = [{"role": "system", "content": SYSTEM}]
    for d in demos:
        msgs += [{"role": "user", "content": user_msg(d)}, {"role": "assistant", "content": d["answer"]}]
    return msgs + [{"role": "user", "content": user_msg(ex)}]


def chat_single(ex, demos):
    shots = "\n\n".join(f"Example {n + 1}\n{user_msg(d)}\nSQL: {d['answer']}" for n, d in enumerate(demos))
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"{shots}\n\nNow answer this one.\n{user_msg(ex)}"}]


def qwen_prompts(ds, layout, demos):
    return [tok_i.apply_chat_template(layout(ex, demos), tokenize=False, add_generation_prompt=True) for ex in ds]


def qwen_generate(ds, layout, demos, desc):
    t0 = time.perf_counter()
    raw = generate_batch(model_i, tok_i, qwen_prompts(ds, layout, demos), max_new_tokens=128, batch_size=16, desc=desc)
    return raw, (time.perf_counter() - t0) / len(ds)


def pick_diverse(region):
    """First short example of each of three query shapes."""
    shapes = [lambda a: "COUNT(" in a.upper() and "JOIN" not in a.upper(),
              lambda a: "WHERE" in a.upper() and "AND" in a.upper() and "(" not in a,
              lambda a: "JOIN" in a.upper()]
    chosen = []
    for shape in shapes:
        ex = next(e for e in region if shape(e["answer"]) and len(e["context"]) < 200 and e not in chosen)
        chosen.append(ex)
    return chosen


rng = random.Random(SEED)
DEMO_SETS = {"random-3": [demo_region[i] for i in rng.sample(range(len(demo_region)), 3)],
             "diverse-3": pick_diverse(demo_region)}
LAYOUTS = {"chat turns": chat_multi, "single message": chat_single}

rows = []
for d_name, demos in DEMO_SETS.items():
    for l_name, layout in LAYOUTS.items():
        raw, _ = qwen_generate(sql_val, layout, demos, f"{d_name}/{l_name}")
        res = evaluate_sql([extract_sql(r) for r in raw], sql_val, f"{d_name} / {l_name}")
        rows.append({**res, "demos": d_name, "layout": l_name})
val_prompt = pd.DataFrame(rows).sort_values(["exact_match", "executable"], ascending=False).reset_index(drop=True)
display(val_prompt)
BEST_DEMOS, BEST_LAYOUT = val_prompt.loc[0, "demos"], val_prompt.loc[0, "layout"]
qwen_val_exact = float(val_prompt.loc[0, "exact_match"])
print("Chosen:", BEST_DEMOS, "/", BEST_LAYOUT)
for d in DEMO_SETS[BEST_DEMOS]:
    print(" demo:", d["question"], "->", d["answer"])

In [ ]:
raw, qwen_sec = qwen_generate(sql_test, LAYOUTS[BEST_LAYOUT], DEMO_SETS[BEST_DEMOS], "test")
SQL_PREDS = {"Qwen2.5-0.5B-Instruct, 3-shot prompt": [extract_sql(r) for r in raw]}
SQL_SEC = {"Qwen2.5-0.5B-Instruct, 3-shot prompt": qwen_sec}
RESULTS_T2.append(evaluate_sql(SQL_PREDS["Qwen2.5-0.5B-Instruct, 3-shot prompt"], sql_test, "Qwen2.5-0.5B-Instruct, 3-shot prompt"))

no_extract = evaluate_sql(raw, sql_test, "Qwen raw replies, no extraction (diagnostic)")
has_fence = np.mean(["```" in r for r in raw])
print(f"Replies containing a code fence: {has_fence:.1%}")
print("Without extraction:", no_extract)
print("With extraction   :", RESULTS_T2[-1])
print("\nExample raw reply:\n", raw[0])

del model_i; clear_gpu()

#### 2. Fine-tuning GPT-2
Each example becomes one text sequence. Only the SQL tokens and the final end-of-text token have labels, so the schema and question are context to read rather than text to learn to write. The end-of-text token teaches the model when to stop.

I try two input formats on 500 training examples and validate on `sql_val`. Format A uses section headers like the class demo. Format B writes the input as SQL comments, which may suit a model that has seen code. The better format is then used for the 500- and 2,000-example runs, with the same hyperparameters. The 500 examples are the first 500 of the 2,000.

Examples whose prompt and answer exceed 256 tokens are dropped from training only.

In [ ]:
tok_gpt = AutoTokenizer.from_pretrained("openai-community/gpt2")
tok_gpt.pad_token = tok_gpt.eos_token

FORMATS = {
    "A: section headers": lambda ex: f"### Schema:\n{ex['context']}\n\n### Question:\n{ex['question']}\n\n### SQL:\n",
    "B: SQL comments":    lambda ex: f"/* Schema */\n{ex['context']};\n/* Question: {ex['question']} */\n",
}
HP = dict(per_device_train_batch_size=16, per_device_eval_batch_size=32, num_train_epochs=10, learning_rate=1e-4,
          weight_decay=0.01, lr_scheduler_type="cosine")


def n_tokens(ex, fmt):
    return len(tok_gpt(fmt(ex) + ex["answer"], add_special_tokens=False).input_ids) + 1


def fits(ex):
    return max(n_tokens(ex, f) for f in FORMATS.values()) <= MAX_LEN


train_fit = sql_train_pool.select(range(2200)).filter(fits)
val_fit = sql_val.filter(fits)
assert len(train_fit) >= 2000, len(train_fit)
train_2000 = train_fit.select(range(2000))
train_500 = train_2000.select(range(500))
print(f"Training candidates after the length filter: {len(train_fit)} of 2200. Validation for loss: {len(val_fit)}.")


def tokenize(ds, fmt):
    def one(ex):
        p = tok_gpt(fmt(ex), add_special_tokens=False).input_ids
        r = tok_gpt(ex["answer"], add_special_tokens=False).input_ids + [tok_gpt.eos_token_id]
        ids = p + r
        return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": [-100] * len(p) + r}
    return ds.map(one, remove_columns=ds.column_names)


def train_gpt2(train_raw, fmt, name):
    set_seed(SEED)
    out_dir = f"./gpt2-sql-{name}"
    model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(device)
    model.config.pad_token_id = tok_gpt.eos_token_id
    total_steps = int(np.ceil(len(train_raw) / HP["per_device_train_batch_size"])) * HP["num_train_epochs"]
    args = TrainingArguments(output_dir=out_dir, warmup_steps=int(0.1 * total_steps), eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
                             load_best_model_at_end=True, metric_for_best_model="loss", greater_is_better=False,
                             logging_strategy="epoch", fp16=(device == "cuda"), report_to="none", seed=SEED,
                             disable_tqdm=True, **HP)
    trainer = Trainer(model=model, args=args, train_dataset=tokenize(train_raw, fmt), eval_dataset=tokenize(val_fit, fmt),
                      data_collator=DataCollatorForSeq2Seq(tok_gpt, padding=True, label_pad_token_id=-100))
    t0 = time.perf_counter()
    trainer.train()
    train_sec = time.perf_counter() - t0
    history = [l for l in trainer.state.log_history if "eval_loss" in l]
    shutil.rmtree(out_dir, ignore_errors=True)
    return model.eval(), train_sec, history


def gpt2_predict(model, ds, fmt, desc):
    prompts = [fmt(ex) for ex in ds]
    longest = max(len(tok_gpt(p, add_special_tokens=False).input_ids) for p in prompts)
    assert longest + MAX_NEW_SQL <= 1024, longest
    t0 = time.perf_counter()
    outs = generate_batch(model, tok_gpt, prompts, max_new_tokens=MAX_NEW_SQL, batch_size=16, stop_strings=["\n"], desc=desc)
    return outs, (time.perf_counter() - t0) / len(ds)


rows = []
for f_name, fmt in FORMATS.items():
    model, train_sec, _ = train_gpt2(train_500, fmt, "format")
    preds, _ = gpt2_predict(model, sql_val, fmt, f_name)
    rows.append({**evaluate_sql(preds, sql_val, f_name), "train_sec": round(train_sec, 1)})
    del model; clear_gpu()
val_fmt = pd.DataFrame(rows).sort_values(["exact_match", "executable"], ascending=False).reset_index(drop=True)
display(val_fmt)
BEST_FMT = FORMATS[val_fmt.loc[0, "system"]]
print("Chosen format:", val_fmt.loc[0, "system"])
print("\nFormatted example:\n" + BEST_FMT(train_500[0]) + train_500[0]["answer"])

In [ ]:
RUNS = {"GPT-2 fine-tuned, 500 examples": ("500", train_500), "GPT-2 fine-tuned, 2,000 examples": ("2000", train_2000)}
TRAIN_SEC, HISTORY, VAL_EXACT = {}, {}, {"Qwen2.5-0.5B-Instruct, 3-shot prompt": qwen_val_exact}

for name, (tag, data) in RUNS.items():
    model, TRAIN_SEC[name], HISTORY[name] = train_gpt2(data, BEST_FMT, tag)
    val_preds, _ = gpt2_predict(model, sql_val, BEST_FMT, f"val {tag}")
    VAL_EXACT[name] = evaluate_sql(val_preds, sql_val, name)["exact_match"]
    SQL_PREDS[name], SQL_SEC[name] = gpt2_predict(model, sql_test, BEST_FMT, f"test {tag}")
    RESULTS_T2.append(evaluate_sql(SQL_PREDS[name], sql_test, name))
    print(f"{name}: train {TRAIN_SEC[name]:.0f} s | best epoch "
          f"{min(HISTORY[name], key=lambda l: l['eval_loss'])['epoch']:.0f} | {RESULTS_T2[-1]}")
    del model; clear_gpu()

fig, ax = plt.subplots(figsize=(6.4, 3.6))
for colour, (name, hist) in zip(["#2a78b5", "#c8553d"], HISTORY.items()):
    x = [l["epoch"] for l in hist]; y = [l["eval_loss"] for l in hist]
    ax.plot(x, y, marker="o", markersize=5, linewidth=2, color=colour, label=name)
    ax.annotate(name.split(", ")[1], (x[-1], y[-1]), xytext=(6, 0), textcoords="offset points", fontsize=9, va="center")
ax.set_xlabel("epoch"); ax.set_ylabel("validation loss (SQL tokens)")
ax.set_title("Task 2: validation loss during fine-tuning", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25); ax.legend(frameon=False, fontsize=9)
ax.set_xlim(right=max(x) + 1.8)
plt.tight_layout(); plt.show()

#### 3. Error analysis
The best system is chosen by validation exact match, and I take its **first 10** test mistakes in dataset order so they are not cherry-picked. The labels are assigned by hand in the next section.

In [ ]:
BEST_SQL = max(VAL_EXACT, key=VAL_EXACT.get)
print("Validation exact match:", {k: round(v, 3) for k, v in VAL_EXACT.items()})
print("Best system (selected on validation):", BEST_SQL, "\n")

best_preds = SQL_PREDS[BEST_SQL]
mistakes = [i for i in range(len(sql_test)) if normalize_sql(best_preds[i]) != normalize_sql(sql_test[i]["answer"])]
print(f"Exact-match mistakes: {len(mistakes)} of {len(sql_test)}; "
      f"of those, {sum(executes(best_preds[i], sql_test[i]['context']) for i in mistakes)} still execute.\n")
for n, i in enumerate(mistakes[:10], 1):
    ex = sql_test[i]
    print(f"[{n}] test #{i} | executes: {executes(best_preds[i], ex['context'])}")
    print("  SCHEMA  :", ex["context"]); print("  QUESTION:", ex["question"])
    print("  GOLD    :", ex["answer"]); print("  PRED    :", best_preds[i]); print()

### Task 2: Results and analysis ✍️

_Results table, plots, and answers to the analysis questions go here._

In [ ]:
def paired_bootstrap(correct_a, correct_b, n=10000, seed=SEED):
    """Metric difference a - b with a 95% bootstrap interval, resampling test examples."""
    a, b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), (n, len(a)))
    diffs = (a[idx] - b[idx]).mean(1)
    return (a - b).mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)


table_t2 = pd.DataFrame(RESULTS_T2)
table_t2["sec_per_query"] = table_t2["system"].map(SQL_SEC)
table_t2["train_sec"] = table_t2["system"].map(TRAIN_SEC)
display(table_t2.round({"exact_match": 3, "executable": 3, "sec_per_query": 4, "train_sec": 1}))


def exact(name):
    return [normalize_sql(p) == normalize_sql(g) for p, g in zip(SQL_PREDS[name], sql_test["answer"])]


Q, G5, G2 = ("Qwen2.5-0.5B-Instruct, 3-shot prompt", "GPT-2 fine-tuned, 500 examples", "GPT-2 fine-tuned, 2,000 examples")
rows = []
for a, b in [(G2, G5), (G2, Q), (G5, Q)]:
    d, lo, hi = paired_bootstrap(exact(a), exact(b))
    rows.append({"A": a, "B": b, "exact match A - B": round(d, 3), "95% CI low": round(lo, 3), "95% CI high": round(hi, 3)})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
short = table_t2["system"].str.replace("Qwen2.5-0.5B-Instruct, ", "Qwen ", regex=False).str.replace("GPT-2 fine-tuned, ", "GPT-2, ", regex=False)
for ax, metric, title in zip(axes, ["exact_match", "executable"], ["Exact match", "Executable"]):
    bars = ax.barh(short, table_t2[metric], color="#2a78b5", height=0.6)
    ax.bar_label(bars, labels=[f"{v:.3f}" for v in table_t2[metric]], padding=4, fontsize=9)
    ax.set_xlim(0, 1.12); ax.set_title(title, loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="x", alpha=0.25); ax.set_axisbelow(True)
axes[0].invert_yaxis()
fig.suptitle("Task 2: results on sql_test (200 examples)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()